In [0]:
dbutils.widgets.text("catalog", "fk")
catalog = dbutils.widgets.get("catalog")
c = catalog

In [0]:


spark.sql(f"GRANT USE CATALOG ON CATALOG {c} TO `fk_analysts`")
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {c}.gold TO `fk_analysts`")
spark.sql(f"GRANT USE CATALOG ON CATALOG {c} TO `fk_engineers`")

for s in ["bronze", "silver", "gold", "ops"]:
    spark.sql(f"GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA {c}.{s} TO `fk_engineers`")
    spark.sql(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {c}.landing.raw TO `fk_engineers`")      
            


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {c}.ops.user_entitlements (
  user_email STRING, allowed_city STRING, can_view_pii BOOLEAN)
COMMENT 'Per-user city access and PII visibility used by column masks and the row filter'
""")

me = spark.sql("SELECT current_user()").first()[0]

def set_entitlement(city, pii):
    spark.sql(f"DELETE FROM {c}.ops.user_entitlements")
    spark.sql(f"INSERT INTO {c}.ops.user_entitlements VALUES ('{me}', '{city}', {str(pii).upper()})")

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {c}.ops.city_filter(city STRING) RETURNS BOOLEAN
RETURN {is_admin} OR EXISTS (
  SELECT 1 FROM {c}.ops.user_entitlements e
  WHERE e.user_email = current_user() AND (e.allowed_city = 'ALL' OR e.allowed_city = city))
""")

spark.sql(f"ALTER TABLE {c}.gold.agg_daily_store_sales SET ROW FILTER {c}.ops.city_filter ON (city)")

In [0]:
set_entitlement("Jaipur", False)

jaipur_rows = spark.sql(f"SELECT count(*) FROM {c}.gold.agg_daily_store_sales").first()[0]
cust = spark.sql(f"""SELECT email, phone FROM {c}.gold.dim_customer
                     WHERE customer_id = 'C0001' AND is_current""").first()

(spark.createDataFrame(
    [("ROW_FILTER_ROW_COUNT_JAIPUR", str(jaipur_rows)),
     ("MASKED_EMAIL_C0001", cust["email"]),
     ("MASKED_PHONE_C0001", cust["phone"])],
    "evidence_name STRING, observed_value STRING")
 .write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{c}.ops.governance_evidence"))

spark.sql(f"COMMENT ON TABLE {c}.ops.governance_evidence IS 'Observed effect of the row filter and masks for a Jaipur analyst without PII access'")
spark.table(f"{c}.ops.governance_evidence").show(truncate=False)

In [0]:
set_entitlement("ALL", False)
spark.table(f"{c}.ops.user_entitlements").show()   